In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

# Synthetic Hospital Occupancy Dataset
# Step 2: Environment setup

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility configuration
SEED = 20261001
rng = np.random.default_rng(SEED)

print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Matplotlib version:", plt.matplotlib.__version__)
print("Random seed:", SEED)
print("Environment setup successful!")


In [ ]:
# Step 3: Simulation configuration

# Reproducibility
SEED = 20261001

# Hospital settings
CAPACITY = 600
TARGET_OCCUPANCY = 480

# Patient arrivals
BASELINE_ARRIVALS = 30

# Length of stay (LOS): Gamma distribution
LOS_SHAPE = 4
LOS_SCALE = 4  # days

# Annual seasonality
SEASONAL_AMPLITUDE = 0.08
SEASONAL_PEAK_MONTH = 1
SEASONAL_PEAK_DAY = 15

# Weekly admission multipliers (Monday = 0)
WEEKDAY_MULTIPLIERS = {
    0: 1.10,  # Monday
    1: 1.05,  # Tuesday
    2: 1.00,  # Wednesday
    3: 1.00,  # Thursday
    4: 0.95,  # Friday
    5: 0.95,  # Saturday
    6: 0.95,  # Sunday
}

# Simulation period
WARMUP_DAYS = 180
STUDY_START = "2021-01-01"
STUDY_END = "2024-12-31"

# Derived dates
study_dates = pd.date_range(
    start=STUDY_START,
    end=STUDY_END,
    freq="D"
)

TOTAL_STUDY_DAYS = len(study_dates)
TOTAL_SIMULATION_DAYS = WARMUP_DAYS + TOTAL_STUDY_DAYS

print("Configuration loaded successfully.")
print("Study days:", TOTAL_STUDY_DAYS)
print("Warm-up days:", WARMUP_DAYS)
print("Total simulated days:", TOTAL_SIMULATION_DAYS)
print("Mean theoretical LOS:", LOS_SHAPE * LOS_SCALE, "days")
print("Mean weekday multiplier:",
      np.mean(list(WEEKDAY_MULTIPLIERS.values())))


In [ ]:
# Step 4: Generate daily patient arrivals

# Reset the random generator to ensure reproducibility
rng = np.random.default_rng(SEED)

# Create dates covering the warm-up and study periods
simulation_start = (
    pd.Timestamp(STUDY_START) - pd.Timedelta(days=WARMUP_DAYS)
)

simulation_dates = pd.date_range(
    start=simulation_start,
    end=STUDY_END,
    freq="D"
)

# Calculate weekday multipliers
weekday_factors = np.array([
    WEEKDAY_MULTIPLIERS[d.weekday()]
    for d in simulation_dates
])

# Calculate annual seasonal multipliers
peak_date = pd.Timestamp(
    year=2021,
    month=SEASONAL_PEAK_MONTH,
    day=SEASONAL_PEAK_DAY
)

elapsed_days = np.array([
    (d - peak_date).days
    for d in simulation_dates
])

annual_period = 365.2425

seasonal_factors = (
    1 + SEASONAL_AMPLITUDE
    * np.cos(2 * np.pi * elapsed_days / annual_period)
)

# Calculate expected arrivals for each day
expected_arrivals = (
    BASELINE_ARRIVALS
    * weekday_factors
    * seasonal_factors
)

# Generate actual arrivals using the Poisson distribution
daily_arrivals = rng.poisson(expected_arrivals)

# Store the results in a DataFrame
arrivals_df = pd.DataFrame({
    "date": simulation_dates,
    "weekday": simulation_dates.day_name(),
    "weekday_multiplier": weekday_factors,
    "seasonal_multiplier": seasonal_factors,
    "expected_arrivals": expected_arrivals,
    "actual_arrivals": daily_arrivals,
})

# Identify warm-up and study periods
arrivals_df["period"] = np.where(
    arrivals_df["date"] < pd.Timestamp(STUDY_START),
    "warm-up",
    "study"
)

print("Arrival generation complete.")
print("Total simulated days:", len(arrivals_df))
print("Total simulated arrivals:", arrivals_df["actual_arrivals"].sum())
print("Mean daily arrivals:", round(arrivals_df["actual_arrivals"].mean(), 2))
print("Mean expected arrivals:", round(arrivals_df["expected_arrivals"].mean(), 2))

display(arrivals_df.head(10))


In [ ]:

# Step 4 validation: inspect arrival patterns

study_arrivals = arrivals_df[
    arrivals_df["period"] == "study"
].copy()

# Average arrivals by weekday
weekday_summary = (
    study_arrivals.groupby("weekday")["actual_arrivals"]
    .agg(["mean", "std", "count"])
    .reindex([
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    ])
)

print("Average daily arrivals by weekday:")
display(weekday_summary.round(2))

# Plot arrivals over the first 90 study days
first_90_days = study_arrivals.head(90)

plt.figure(figsize=(12, 4))
plt.plot(
    first_90_days["date"],
    first_90_days["actual_arrivals"],
    label="Actual arrivals"
)
plt.plot(
    first_90_days["date"],
    first_90_days["expected_arrivals"],
    label="Expected arrivals",
    linewidth=2
)
plt.title("Daily Patient Arrivals: First 90 Study Days")
plt.xlabel("Date")
plt.ylabel("Number of arrivals")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

# Step 5: Generate individual patient records

# Expand daily arrival counts into individual admission dates
admission_dates = np.repeat(
    arrivals_df["date"].to_numpy(),
    arrivals_df["actual_arrivals"].to_numpy()
)

# Generate Gamma-distributed lengths of stay
# Use the same RNG AFTER arrival generation to preserve the
# complete sequence of random draws for reproducibility.
raw_los = rng.gamma(
    shape=LOS_SHAPE,
    scale=LOS_SCALE,
    size=len(admission_dates)
)

# Round to whole days and enforce a minimum stay of one day
los_days = np.maximum(1, np.rint(raw_los).astype(int))

# Create patient-level records
patients_df = pd.DataFrame({
    "patient_id": [
        f"P{i:06d}" for i in range(1, len(admission_dates) + 1)
    ],
    "admission_date": pd.to_datetime(admission_dates),
    "los_days": los_days
})

# Discharge date is exclusive for occupancy counting
patients_df["discharge_date"] = (
    patients_df["admission_date"]
    + pd.to_timedelta(patients_df["los_days"], unit="D")
)

print("Patient generation complete.")
print("Total patients:", len(patients_df))
print("Mean LOS:", round(patients_df["los_days"].mean(), 3), "days")
print("Median LOS:", round(patients_df["los_days"].median(), 2), "days")
print("Minimum LOS:", patients_df["los_days"].min(), "days")
print("Maximum LOS:", patients_df["los_days"].max(), "days")

display(patients_df.head(10))


In [ ]:
# Step 5 validation: inspect the LOS distribution

print("LOS descriptive statistics:")
display(patients_df["los_days"].describe().round(3))

# Proportion of patients in each LOS range
los_groups = pd.cut(
    patients_df["los_days"],
    bins=[0, 3, 7, 14, 30, np.inf],
    labels=[
        "1–3 days",
        "4–7 days",
        "8–14 days",
        "15–30 days",
        "31+ days"
    ]
)

los_summary = (
    los_groups.value_counts(sort=False)
    .rename("patient_count")
    .to_frame()
)

los_summary["percentage"] = (
    los_summary["patient_count"] / len(patients_df) * 100
)

print("Patients by LOS range:")
display(los_summary.round(2))

# Plot the LOS distribution
plt.figure(figsize=(10, 4))
plt.hist(
    patients_df["los_days"],
    bins=range(1, patients_df["los_days"].max() + 2),
    edgecolor="black"
)
plt.title("Distribution of Patient Length of Stay")
plt.xlabel("Length of stay (days)")
plt.ylabel("Number of patients")
plt.tight_layout()
plt.show()


In [ ]:
# Step 6: Calculate daily occupancy from patient records

# Create a difference array to track admissions and discharges.
# A +1 marks an admission; a -1 marks a discharge.
n_days = len(simulation_dates)
occupancy_changes = np.zeros(n_days + 1, dtype=int)

# Convert dates into integer positions on the simulation timeline
admission_indices = (
    patients_df["admission_date"] - simulation_start
).dt.days.to_numpy()

discharge_indices = (
    patients_df["discharge_date"] - simulation_start
).dt.days.to_numpy()

# Count admissions
np.add.at(
    occupancy_changes,
    admission_indices,
    1
)

# Count discharges that occur within the simulation boundary
valid_discharges = (
    (discharge_indices >= 0)
    & (discharge_indices <= n_days)
)

np.add.at(
    occupancy_changes,
    discharge_indices[valid_discharges],
    -1
)

# Cumulative sum gives the occupied beds on each day
occupancy_values = np.cumsum(occupancy_changes[:-1])

# Create the complete daily occupancy series
occupancy_df = pd.DataFrame({
    "date": simulation_dates,
    "occupied_beds": occupancy_values
})

occupancy_df["capacity"] = CAPACITY
occupancy_df["occupancy_rate"] = (
    occupancy_df["occupied_beds"] / CAPACITY
)

occupancy_df["capacity_exceeded"] = (
    occupancy_df["occupied_beds"] > CAPACITY
)

occupancy_df["period"] = np.where(
    occupancy_df["date"] < pd.Timestamp(STUDY_START),
    "warm-up",
    "study"
)

# Keep only the official four-year study period for analysis
study_occupancy = occupancy_df[
    occupancy_df["period"] == "study"
].copy()

print("Occupancy calculation complete.")
print("Study observations:", len(study_occupancy))
print("Mean occupied beds:",
      round(study_occupancy["occupied_beds"].mean(), 2))
print("Mean occupancy rate:",
      round(study_occupancy["occupancy_rate"].mean() * 100, 2), "%")
print("Maximum occupied beds:",
      study_occupancy["occupied_beds"].max())
print("Days exceeding capacity:",
      int(study_occupancy["capacity_exceeded"].sum()))

display(study_occupancy.head(10))


In [ ]:
# Step 6 validation: occupancy summary and visualisation

print("Occupancy descriptive statistics:")
display(
    study_occupancy["occupied_beds"]
    .describe()
    .round(3)
    .to_frame()
)

# Plot the full four-year occupancy series
plt.figure(figsize=(14, 5))
plt.plot(
    study_occupancy["date"],
    study_occupancy["occupied_beds"],
    linewidth=0.8,
    label="Daily occupancy"
)

plt.axhline(
    TARGET_OCCUPANCY,
    linestyle="--",
    label="Target occupancy (480)"
)

plt.axhline(
    CAPACITY,
    linestyle=":",
    label="Hospital capacity (600)"
)

plt.title("Simulated Daily Hospital Occupancy (2021–2024)")
plt.xlabel("Date")
plt.ylabel("Occupied beds")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Step 7: Quantitative temporal validation

# Ensure observations are sorted chronologically
validation_df = study_occupancy.sort_values("date").copy()

series = validation_df["occupied_beds"]

# 1. Overall occupancy statistics
overall_stats = pd.Series({
    "Mean occupancy (beds)": series.mean(),
    "Standard deviation (beds)": series.std(),
    "Coefficient of variation": series.std() / series.mean(),
    "Lag-1 autocorrelation": series.autocorr(lag=1),
    "Lag-7 autocorrelation": series.autocorr(lag=7),
    "Minimum occupancy (beds)": series.min(),
    "Maximum occupancy (beds)": series.max(),
    "Days above capacity": (series > CAPACITY).sum(),
    "Capacity exceedance rate (%)":
        (series > CAPACITY).mean() * 100
})

print("Overall occupancy validation:")
display(overall_stats.to_frame("value").round(4))

# 2. Mean occupancy by weekday
validation_df["weekday"] = validation_df["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_stats = (
    validation_df.groupby("weekday")["occupied_beds"]
    .agg(["mean", "std", "count"])
    .reindex(weekday_order)
)

print("Occupancy by weekday:")
display(weekday_stats.round(2))

# 3. Mean occupancy by calendar month
validation_df["month"] = validation_df["date"].dt.month

monthly_stats = (
    validation_df.groupby("month")["occupied_beds"]
    .agg(["mean", "std", "count"])
)

monthly_stats.index = [
    pd.Timestamp(2000, month, 1).strftime("%B")
    for month in monthly_stats.index
]

print("Occupancy by calendar month:")
display(monthly_stats.round(2))


In [ ]:

# Step 7: Visual inspection of temporal patterns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Weekday pattern
axes[0].bar(
    weekday_stats.index,
    weekday_stats["mean"]
)
axes[0].set_title("Mean Occupancy by Weekday")
axes[0].set_ylabel("Mean occupied beds")
axes[0].tick_params(axis="x", rotation=45)

# Monthly pattern
axes[1].plot(
    monthly_stats.index,
    monthly_stats["mean"],
    marker="o"
)
axes[1].set_title("Mean Occupancy by Calendar Month")
axes[1].set_ylabel("Mean occupied beds")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Step 7.1: Validate the imposed weekday pattern in arrivals

study_arrivals = arrivals_df[
    arrivals_df["period"] == "study"
].copy()

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

arrival_weekday_stats = (
    study_arrivals.groupby("weekday")["actual_arrivals"]
    .agg(["mean", "std", "count"])
    .reindex(weekday_order)
)

print("Actual arrivals by weekday:")
display(arrival_weekday_stats.round(3))

# Compare actual mean arrivals with the baseline
arrival_weekday_stats["difference_from_baseline"] = (
    arrival_weekday_stats["mean"] - BASELINE_ARRIVALS
)

print("Difference from the baseline of 30 arrivals/day:")
display(
    arrival_weekday_stats[
        ["mean", "difference_from_baseline"]
    ].round(3)
)


In [ ]:
# Step 7.2: Compare monthly arrivals and occupancy

monthly_arrival_check = study_arrivals.copy()
monthly_arrival_check["month"] = (
    monthly_arrival_check["date"].dt.month
)

monthly_arrivals = (
    monthly_arrival_check.groupby("month")
    .agg(
        mean_actual_arrivals=("actual_arrivals", "mean"),
        mean_expected_arrivals=("expected_arrivals", "mean"),
    )
)

monthly_occupancy = (
    validation_df.groupby(validation_df["date"].dt.month)
    ["occupied_beds"].mean()
    .rename("mean_occupied_beds")
)

monthly_check = monthly_arrivals.join(monthly_occupancy)

monthly_check.index = [
    pd.Timestamp(2000, m, 1).strftime("%B")
    for m in monthly_check.index
]

print("Monthly arrivals and occupancy:")
display(monthly_check.round(2))

# Plot expected arrivals against occupancy
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.plot(
    monthly_check.index,
    monthly_check["mean_expected_arrivals"],
    marker="o",
    label="Expected arrivals/day"
)
ax1.plot(
    monthly_check.index,
    monthly_check["mean_actual_arrivals"],
    marker="o",
    label="Actual arrivals/day"
)

ax1.set_ylabel("Arrivals per day")
ax1.set_title("Monthly Arrival Patterns")
ax1.tick_params(axis="x", rotation=45)
ax1.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Step 8: Final dataset consistency checks

# Check date coverage and duplicate dates
assert len(study_occupancy) == 1461
assert study_occupancy["date"].is_unique
assert study_occupancy["date"].min() == pd.Timestamp("2021-01-01")
assert study_occupancy["date"].max() == pd.Timestamp("2024-12-31")

# Check occupancy values
assert (study_occupancy["occupied_beds"] >= 0).all()
assert study_occupancy["occupied_beds"].notna().all()

# Check patient LOS
assert (patients_df["los_days"] >= 1).all()
assert patients_df["patient_id"].is_unique

# Confirm the reported capacity exceedances
capacity_exceedance_days = int(
    (study_occupancy["occupied_beds"] > CAPACITY).sum()
)

# Create the final daily dataset
final_dataset = study_occupancy[
    ["date", "occupied_beds", "occupancy_rate"]
].copy()

final_dataset["occupancy_rate_percent"] = (
    final_dataset["occupancy_rate"] * 100
)

print("All consistency checks passed.")
print("Final dataset shape:", final_dataset.shape)
print("Mean occupancy:", round(final_dataset["occupied_beds"].mean(), 3))
print("Standard deviation:", round(final_dataset["occupied_beds"].std(), 3))
print("Capacity-exceedance days:", capacity_exceedance_days)

display(final_dataset.head())


In [ ]:
# Step 8.1: Export reproducible simulation outputs

import json
from pathlib import Path

# Create an output folder beside the notebook
OUTPUT_DIR = PROJECT_ROOT / "synthetic_hospital_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Reset the daily dataset index before exporting
final_dataset = final_dataset.reset_index(drop=True)

# Export the main forecasting dataset
final_dataset.to_csv(
    OUTPUT_DIR / "hospital_occupancy_daily.csv",
    index=False
)

# Export patient-level records
patients_df.to_csv(
    OUTPUT_DIR / "hospital_patients.csv",
    index=False
)

# Export daily arrival records
arrivals_df.to_csv(
    OUTPUT_DIR / "hospital_arrivals.csv",
    index=False
)

# Document the configuration
config = {
    "seed": SEED,
    "capacity": CAPACITY,
    "target_occupancy": TARGET_OCCUPANCY,
    "baseline_arrivals": BASELINE_ARRIVALS,
    "los_distribution": "Rounded Gamma, minimum 1 day",
    "los_shape": LOS_SHAPE,
    "los_scale_days": LOS_SCALE,
    "seasonal_amplitude": SEASONAL_AMPLITUDE,
    "seasonal_peak_month": SEASONAL_PEAK_MONTH,
    "seasonal_peak_day": SEASONAL_PEAK_DAY,
    "weekday_multipliers": {
        "Monday": 1.10,
        "Tuesday": 1.05,
        "Wednesday": 1.00,
        "Thursday": 1.00,
        "Friday": 0.95,
        "Saturday": 0.95,
        "Sunday": 0.95
    },
    "warmup_days": WARMUP_DAYS,
    "study_start": STUDY_START,
    "study_end": STUDY_END,
    "capacity_rule": "Occupancy uncapped; exceedances recorded",
    "python_version": sys.version.split()[0],
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}

with open(
    OUTPUT_DIR / "hospital_simulation_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=4)

print("Export complete. Files saved in:", OUTPUT_DIR.resolve())

for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{file_path.name}: {file_path.stat().st_size:,} bytes")


In [ ]:
# Verify that the exported daily dataset can be read back

exported_df = pd.read_csv(
    OUTPUT_DIR / "hospital_occupancy_daily.csv",
    parse_dates=["date"]
)

assert len(exported_df) == 1461
assert exported_df["date"].is_unique
assert exported_df["occupied_beds"].notna().all()
assert exported_df["occupied_beds"].min() >= 0

print("Export verification passed.")
print("Rows:", len(exported_df))
print("Date range:",
      exported_df["date"].min().date(),
      "to",
      exported_df["date"].max().date())
print("Mean occupancy:",
      round(exported_df["occupied_beds"].mean(), 3))


In [ ]:
# Step 9.2: Preserve the current output as a reproducibility baseline

import hashlib
import shutil

baseline_path = OUTPUT_DIR / "hospital_occupancy_daily_baseline.csv"
source_path = OUTPUT_DIR / "hospital_occupancy_daily.csv"

shutil.copy2(source_path, baseline_path)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

baseline_hash = sha256_file(baseline_path)

print("Baseline saved:", baseline_path)
print("SHA-256:", baseline_hash)


In [ ]:
# Step 9.4: Compare fresh output with the preserved baseline

fresh_path = OUTPUT_DIR / "hospital_occupancy_daily.csv"

fresh_hash = sha256_file(fresh_path)
baseline_hash = sha256_file(baseline_path)

print("Baseline SHA-256:", baseline_hash)
print("Fresh SHA-256:   ", fresh_hash)

if fresh_hash == baseline_hash:
    print("PASS: The exported CSV is byte-for-byte reproducible.")
else:
    print("CHECK: The CSV differs from the baseline.")
    print("Compare the datasets to identify the cause.")


In [ ]:

# Step 10: Independent reproducibility verification

# Original hash recorded before the fresh-run comparison
EXPECTED_SHA256 = (
    "5b23295a14e8572f6bbbf55cb6793336"
    "a4d9328878880c182b915e437b92b8c3"
)

fresh_path = OUTPUT_DIR / "hospital_occupancy_daily.csv"
fresh_hash = sha256_file(fresh_path)

print("Expected SHA-256:", EXPECTED_SHA256)
print("Current SHA-256: ", fresh_hash)

if fresh_hash == EXPECTED_SHA256:
    print("PASS: Current CSV matches the independently recorded hash.")
else:
    print("FAIL: CSV differs from the recorded reference.")


In [ ]:
# Step 11: Formal dataset validation report

# Use only patients admitted during the official study period
study_patients = patients_df[
    (patients_df["admission_date"] >= pd.Timestamp(STUDY_START))
    & (patients_df["admission_date"] <= pd.Timestamp(STUDY_END))
].copy()

occupancy = study_occupancy["occupied_beds"]

# 1. Overall occupancy metrics
validation_report = {
    "study_start": str(study_occupancy["date"].min().date()),
    "study_end": str(study_occupancy["date"].max().date()),
    "study_days": int(len(study_occupancy)),
    "capacity_beds": int(CAPACITY),
    "target_occupancy_beds": int(TARGET_OCCUPANCY),
    "mean_occupancy_beds": float(occupancy.mean()),
    "std_occupancy_beds": float(occupancy.std()),
    "coefficient_of_variation": float(
        occupancy.std() / occupancy.mean()
    ),
    "minimum_occupancy_beds": int(occupancy.min()),
    "maximum_occupancy_beds": int(occupancy.max()),
    "mean_occupancy_rate_percent": float(
        occupancy.mean() / CAPACITY * 100
    ),
    "lag1_autocorrelation": float(occupancy.autocorr(lag=1)),
    "lag7_autocorrelation": float(occupancy.autocorr(lag=7)),
    "capacity_exceedance_days": int((occupancy > CAPACITY).sum()),
    "capacity_exceedance_percent": float(
        (occupancy > CAPACITY).mean() * 100
    ),
    "study_patient_count": int(len(study_patients)),
    "study_patient_mean_los_days": float(
        study_patients["los_days"].mean()
    ),
    "study_patient_median_los_days": float(
        study_patients["los_days"].median()
    ),
    "study_patient_los_std_days": float(
        study_patients["los_days"].std()
    ),
    "study_patient_minimum_los_days": int(
        study_patients["los_days"].min()
    ),
    "study_patient_maximum_los_days": int(
        study_patients["los_days"].max()
    ),
    "random_seed": int(SEED),
}

# 2. Weekday summaries
weekday_arrival_report = (
    study_arrivals.groupby("weekday")["actual_arrivals"]
    .agg(["mean", "std", "count"])
    .reindex(weekday_order)
)

weekday_occupancy_report = (
    validation_df.groupby("weekday")["occupied_beds"]
    .agg(["mean", "std", "count"])
    .reindex(weekday_order)
)

# 3. Monthly summaries
monthly_arrival_report = (
    study_arrivals.assign(
        month=study_arrivals["date"].dt.month
    )
    .groupby("month")
    .agg(
        mean_actual_arrivals=("actual_arrivals", "mean"),
        mean_expected_arrivals=("expected_arrivals", "mean"),
    )
)

monthly_occupancy_report = (
    validation_df.assign(
        month=validation_df["date"].dt.month
    )
    .groupby("month")["occupied_beds"]
    .agg(["mean", "std", "count"])
)

# 4. Export report files
OUTPUT_DIR.mkdir(exist_ok=True)

with open(
    OUTPUT_DIR / "hospital_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(validation_report, f, indent=4)

pd.DataFrame(
    [validation_report]
).to_csv(
    OUTPUT_DIR / "hospital_validation_summary.csv",
    index=False
)

weekday_arrival_report.to_csv(
    OUTPUT_DIR / "validation_weekday_arrivals.csv"
)

weekday_occupancy_report.to_csv(
    OUTPUT_DIR / "validation_weekday_occupancy.csv"
)

monthly_arrival_report.to_csv(
    OUTPUT_DIR / "validation_monthly_arrivals.csv"
)

monthly_occupancy_report.to_csv(
    OUTPUT_DIR / "validation_monthly_occupancy.csv"
)

print("Validation report generated successfully.")
print("\nKey results:")

for key, value in validation_report.items():
    print(f"{key}: {value}")

print("\nReport directory:", OUTPUT_DIR.resolve())


In [ ]:
# STEP 1: Audit the synthetic hospital dataset generator

print("=" * 55)
print("SYNTHETIC DATASET GENERATOR AUDIT")
print("=" * 55)

# 1. Configuration checks
assert SEED == 20261001
assert CAPACITY == 600
assert BASELINE_ARRIVALS == 30
assert LOS_SHAPE == 4 and LOS_SCALE == 4
assert WARMUP_DAYS == 180
assert len(study_dates) == 1461

print("\n1. Configuration: PASS")

# 2. Check weekday multipliers
weekday_mean = np.mean(list(WEEKDAY_MULTIPLIERS.values()))

assert np.isclose(weekday_mean, 1.0)
assert len(WEEKDAY_MULTIPLIERS) == 7

print("2. Weekday multipliers: PASS")
print("   Mean multiplier:", weekday_mean)

# 3. Check the expected-arrival equation
expected_check = (
    BASELINE_ARRIVALS
    * arrivals_df["weekday_multiplier"].to_numpy()
    * arrivals_df["seasonal_multiplier"].to_numpy()
)

assert np.allclose(
    expected_check,
    arrivals_df["expected_arrivals"].to_numpy()
)

assert (
    arrivals_df["actual_arrivals"].ge(0).all()
)

print("3. Arrival equation: PASS")

# 4. Check seasonal-factor boundaries
seasonal_min = arrivals_df["seasonal_multiplier"].min()
seasonal_max = arrivals_df["seasonal_multiplier"].max()

print("4. Seasonal factor range:")
print("   Minimum:", round(seasonal_min, 4))
print("   Maximum:", round(seasonal_max, 4))

# 5. Check patient records
assert patients_df["patient_id"].is_unique
assert patients_df["los_days"].min() >= 1

assert (
    patients_df["discharge_date"]
    > patients_df["admission_date"]
).all()

assert (
    patients_df["discharge_date"]
    == patients_df["admission_date"]
       + pd.to_timedelta(patients_df["los_days"], unit="D")
).all()

print("5. Patient records: PASS")

# 6. Check daily occupancy data
assert len(study_occupancy) == 1461
assert study_occupancy["date"].is_unique
assert study_occupancy["occupied_beds"].notna().all()
assert (study_occupancy["occupied_beds"] >= 0).all()

print("6. Occupancy records: PASS")

# 7. Check that the exported data matches the study series
assert np.array_equal(
    exported_df["occupied_beds"].to_numpy(),
    study_occupancy["occupied_beds"].to_numpy()
)

print("7. Exported data consistency: PASS")

print("\nAUDIT COMPLETE")


In [ ]:

# STEP 2: Consolidated statistical validation

# Official study-period occupancy
occ = study_occupancy["occupied_beds"].astype(float)

# Official study-period patient admissions
study_patient_records = patients_df[
    (patients_df["admission_date"] >= pd.Timestamp(STUDY_START))
    & (patients_df["admission_date"] <= pd.Timestamp(STUDY_END))
]

los = study_patient_records["los_days"].astype(float)

# Weekly admission and occupancy patterns
arrivals_by_weekday = (
    study_arrivals.groupby("weekday")["actual_arrivals"]
    .mean()
    .reindex(weekday_order)
)

occupancy_by_weekday = (
    study_occupancy.assign(
        weekday=study_occupancy["date"].dt.day_name()
    )
    .groupby("weekday")["occupied_beds"]
    .mean()
    .reindex(weekday_order)
)

# Monthly seasonal patterns
arrivals_by_month = (
    study_arrivals.assign(
        month=study_arrivals["date"].dt.month
    )
    .groupby("month")["actual_arrivals"]
    .mean()
)

occupancy_by_month = (
    study_occupancy.assign(
        month=study_occupancy["date"].dt.month
    )
    .groupby("month")["occupied_beds"]
    .mean()
)

# Build a concise validation summary
validation_table = pd.DataFrame({
    "metric": [
        "Study observations",
        "Study-period patient admissions",
        "Mean LOS (days)",
        "Median LOS (days)",
        "LOS standard deviation (days)",
        "Minimum LOS (days)",
        "Maximum LOS (days)",
        "Mean occupancy (beds)",
        "Occupancy standard deviation (beds)",
        "Occupancy coefficient of variation",
        "Minimum occupancy (beds)",
        "Maximum occupancy (beds)",
        "Lag-1 autocorrelation",
        "Lag-7 autocorrelation",
        "Capacity exceedance days",
        "Mean arrivals, Monday",
        "Mean arrivals, Sunday",
        "Highest monthly mean arrivals",
        "Lowest monthly mean arrivals",
        "Highest monthly mean occupancy",
        "Lowest monthly mean occupancy",
    ],
    "value": [
        len(study_occupancy),
        len(study_patient_records),
        los.mean(),
        los.median(),
        los.std(),
        los.min(),
        los.max(),
        occ.mean(),
        occ.std(),
        occ.std() / occ.mean(),
        occ.min(),
        occ.max(),
        occ.autocorr(lag=1),
        occ.autocorr(lag=7),
        int((occ > CAPACITY).sum()),
        arrivals_by_weekday["Monday"],
        arrivals_by_weekday["Sunday"],
        arrivals_by_month.max(),
        arrivals_by_month.min(),
        occupancy_by_month.max(),
        occupancy_by_month.min(),
    ]
})

display(validation_table.round(4))

# Export the consolidated results
validation_table.to_csv(
    OUTPUT_DIR / "consolidated_validation_results.csv",
    index=False
)

# Export weekly and monthly summaries
pd.DataFrame({
    "mean_arrivals": arrivals_by_weekday,
    "mean_occupancy": occupancy_by_weekday
}).to_csv(OUTPUT_DIR / "weekday_validation_results.csv")

pd.DataFrame({
    "mean_arrivals": arrivals_by_month,
    "mean_occupancy": occupancy_by_month
}).to_csv(OUTPUT_DIR / "monthly_validation_results.csv")

print("Validation tables exported successfully.")


In [ ]:

# STEP 3: Final data-quality check

import pandas as pd

dataset_path = OUTPUT_DIR / "hospital_occupancy_daily.csv"
df_check = pd.read_csv(dataset_path, parse_dates=["date"])

print("Dataset shape:", df_check.shape)
print("\nColumn names:")
print(df_check.columns.tolist())

print("\nDate range:")
print(df_check["date"].min(), "to", df_check["date"].max())

print("\nMissing values per column:")
print(df_check.isnull().sum())

print("\nDuplicate dates:", df_check["date"].duplicated().sum())

print(
    "Dates consecutive:",
    df_check["date"].sort_values().diff().dropna().eq(
        pd.Timedelta(days=1)
    ).all()
)

print("\nFirst five rows:")
display(df_check.head())

print("\nLast five rows:")
display(df_check.tail())
